# CAN-Age: actual held-out results

Run `python scripts/download_assets.py` from the repository root first. This notebook reads the released foreground probabilities and summaries.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
summary = json.loads((ROOT / 'results/primary/summary.json').read_text())
oof = pd.read_csv(ROOT / 'results/primary/oof_predictions.csv')
assert len(oof) == 1143 * 12


In [ ]:
records = []
for backbone, variants in summary.items():
    for variant, result in variants.items():
        records.append({'backbone': backbone, 'variant': variant, **{k: result['pooled'][k] for k in ['n', 'accuracy', 'macro_f1', 'ordinal_mae']}})
performance = pd.DataFrame(records)
performance


In [ ]:
performance.pivot(index='backbone', columns='variant', values='accuracy').mul(100).plot.bar(figsize=(9, 4))
plt.ylabel('Held-out accuracy (%)')
plt.tight_layout()


In [ ]:
name = 'densenet121'
rows = oof[(oof.backbone == name) & (oof.variant == 'color_attention')].sort_values('image_index')
probability = rows[[f'prob_age_{age}' for age in range(1, 6)]].to_numpy()
labels = rows.age_label.to_numpy() - 1
predicted = probability.argmax(1)
cm = np.zeros((5, 5), dtype=int)
np.add.at(cm, (labels, predicted), 1)
assert cm.tolist() == summary[name]['color_attention']['pooled']['confusion_matrix']
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Blues')
for i in range(5):
    for j in range(5):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center')
ax.set(xticks=range(5), yticks=range(5), xticklabels=['1','2','3','4','5+'], yticklabels=['1','2','3','4','5+'], xlabel='Predicted winter stage', ylabel='Annotated winter stage')
plt.tight_layout()
